# c_Run_Tools Runner

`run_opensim_pipeline.py`를 노트북에서 셀 단위로 실행하기 위한 작업 노트북입니다.

- 단일 피험자 / 조건 실행
- `--dry-run` 점검
- 세그먼트 단위 제한 실행
- (옵션) 여러 조건 반복 실행

In [ ]:
import os
import sys
import subprocess

# 노트북 실행 위치가 어디든 동작하도록 경로를 정렬
repo_root = os.getcwd()
if not os.path.isdir(os.path.join(repo_root, "Codes")):  # if you run this notebook from other directory, change below
    repo_root = r"C:/Users/ok/Documents/GitHub/BOX"      # (optional)TODO: change to your path

work_dir = os.path.join(repo_root, "Codes", "c_Run_Tools")
os.chdir(work_dir)

if work_dir not in sys.path:
    sys.path.insert(0, work_dir)
codes_dir = os.path.dirname(work_dir)
if codes_dir not in sys.path:
    sys.path.insert(0, codes_dir)


def run_cmd_show_output(cmd):
    """Notebook에서 subprocess 출력을 실시간으로 보여 주는 helper.

    ``capture_output=True`` 는 프로세스가 끝날 때까지 출력을 묶어두므로
    긴 SO/JR 실행 중 진행 상황을 볼 수 없다.  line-by-line streaming +
    ``PYTHONUNBUFFERED=1`` 로 중간 ``[DONE]`` 로그가 바로 보이게 한다.
    """
    print("[RUN]", " ".join(cmd), flush=True)
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}")


print("cwd:", os.getcwd())
print("work_dir:", work_dir)

In [ ]:
from SUB_Info import subjects

print("Available subjects:", list(subjects.keys()))

namecode = "260512_KCH"  # TODO: 필요 시 변경
print("Conditions:", list(subjects[namecode]["conditions"].keys()))

In [ ]:
# 단일 피험자/조건 실행
# namecode = "260306_KTY"     # TODO
condition = "7kg_10bpm"     # TODO
tools = "extload, ik"               # 현재 구현 범위
segments = "1AB,1BC,1CA"                 # 검증용 소량 실행

cmd = [
    sys.executable,
    "run_opensim_pipeline.py",
    "--namecode", namecode,
    "--condition", condition,
    "--tools", tools,
    "--segments", segments,
    # "--dry-run",
]

run_cmd_show_output(cmd)

In [ ]:
# 단일 피험자/조건 실제 실행
# 주의: 실제 파일 생성/실행이 발생합니다.
# namecode = "240124_PJH"            # TODO
condition = "7kg_10bpm_trial1"     # TODO
tools = "extload,ik"
segments = "1U"                    # 처음엔 1개 세그먼트로 권장

cmd = [
    sys.executable,
    "run_opensim_pipeline.py",
    "--namecode", namecode,
    "--condition", condition,
    "--tools", tools,
    "--segments", segments,
    # "--dry-run",
]

run_cmd_show_output(cmd)

In [ ]:
# 여러 조건 일괄 실행 (옵션)
# namecode 는 위 셀에서 설정
conditions = [
    "7kg_10bpm",
    "7kg_16bpm",
    "15kg_10bpm",
    "15kg_16bpm",
]
tools = "extload,ik"

for cond in conditions:
    cmd = [
        sys.executable,
        "run_opensim_pipeline.py",
        "--namecode", namecode,
        "--condition", cond,
        "--tools", tools,
        # "--dry-run",
    ]
    print()
    run_cmd_show_output(cmd)